# 🏠 House Price Prediction – Exploratory Notebook

This notebook walks through:
1. Loading and exploring the California Housing dataset
2. Visualising distributions and correlations
3. Training Linear Regression and Random Forest
4. Comparing model performance
5. Making a sample prediction

> **Run from the project root:**  `jupyter notebook notebooks/exploration.ipynb`

In [ ]:
# Make sure the project root is on the path
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
print('✅ Imports OK')

## 1 · Load Data

In [ ]:
from data.data_loader import load_california_housing, basic_eda

df = load_california_housing()
basic_eda(df)

In [ ]:
df.head()

## 2 · Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['MedHouseVal'], bins=50, color='#C0582A', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Median House Value ($100k)')
axes[0].set_title('Raw Distribution')

axes[1].hist(np.log1p(df['MedHouseVal']), bins=50, color='#4A7C59', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('log(1 + Median House Value)')
axes[1].set_title('Log-Transformed')

plt.suptitle('Target: Median House Value', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3 · Correlation Heatmap

In [ ]:
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            ax=ax, cbar_kws={'shrink': 0.75})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with the target
print('\nTop correlations with MedHouseVal:')
print(corr['MedHouseVal'].sort_values(ascending=False))

## 4 · Train Both Models

In [ ]:
from data.preprocessing import get_features_and_target, split_data
from model.train        import get_models, evaluate_model

X, y = get_features_and_target(df)
X_train, X_test, y_train, y_test = split_data(X, y)

models  = get_models()
results = {}

for name, pipeline in models.items():
    print(f'Training {name}…')
    pipeline.fit(X_train, y_train)
    metrics = evaluate_model(name, pipeline, X_test, y_test)
    results[name] = {'pipeline': pipeline, 'metrics': metrics}

## 5 · Model Comparison

In [ ]:
summary = pd.DataFrame([v['metrics'] for v in results.values()])
summary = summary.set_index('name')
print(summary)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics_to_plot = [('mae','MAE (↓ better)'), ('rmse','RMSE (↓ better)'), ('r2','R² (↑ better)')]
colors = ['#4C72B0', '#DD8452']

for ax, (metric, title) in zip(axes, metrics_to_plot):
    vals  = summary[metric]
    bars  = ax.bar(vals.index, vals.values, color=colors, edgecolor='white')
    ax.set_title(title)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6 · Feature Importance (Random Forest)

In [ ]:
from data.preprocessing import FEATURE_COLS

rf        = results['Random Forest']['pipeline']
importances = rf.named_steps['regressor'].feature_importances_
indices     = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([FEATURE_COLS[i] for i in indices], importances[indices],
       color=sns.color_palette('viridis', len(FEATURE_COLS)), edgecolor='white')
ax.set_title('Random Forest – Feature Importances', fontsize=13, fontweight='bold')
ax.set_ylabel('Importance Score')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 7 · Make a Sample Prediction

In [ ]:
import numpy as np

# Sample house in Los Angeles area
sample = np.array([[3.87, 29.0, 5.43, 1.09, 1015.0, 2.72, 34.05, -118.24]])

best_model = results['Random Forest']['pipeline']
pred_100k  = best_model.predict(sample)[0]

print(f'Predicted median house value : {pred_100k:.3f} × $100k')
print(f'                             = ${pred_100k * 100_000:,.0f}')